## AY25-26 Term 2 CS421 Group Project

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis, entropy
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold

data=np.load("../input/training_batch_with_labels.npz")
print(data)

NpzFile '../input/training_batch_with_labels.npz' with keys: X, y


In [10]:
X=data["X"]
y=data["y"]

print("# of interactions:", X.shape[0])
print("# of anomalous and normal users:", np.count_nonzero(y==1), np.count_nonzero(y==0))


df=pd.DataFrame(X)
labels=pd.DataFrame(y)
df.rename(columns={0:"user",1:"item",2:"rating"},inplace=True)
print("# of items:", df['item'].unique().shape[0])

# of interactions: 177346
# of anomalous and normal users: 100 1000
# of items: 993


In [4]:
df.head(100)

,user,item,rating
0,304,0,3
1,304,1,3
2,304,14,3
3,304,17,4
4,304,19,4
...,...,...,...
95,304,824,3
96,304,826,3
97,304,828,4
98,304,847,3


In [11]:
labels.rename(columns={0:"user",1:"label"},inplace=True)

In [12]:
labels.head(10)

,user,label
0,100,0
1,101,0
2,102,0
3,103,0
4,104,0
5,105,0
6,106,0
7,107,0
8,108,0
9,109,0


## Feature Engineering

The objective is to derive user-level behavioural features from the interaction dataset (user, item, rating) so that the model can learn behavioural patterns that distinguish anomalous users from normal users.

Since anomaly detection occurs at the user level, all features must summarize a user's rating behaviour across items.

Derive BaseLine user level features.

In [28]:
def build_user_baseline(df):

    # enforce schema
    df = df.copy()
    df.columns = ["user", "item", "rating"]

    print(f'Shape of data before accounting for duplicate ratings: {df.shape}')
    # ensure there are no duplicate ratings for a particular item by the same user
    df = df.groupby(["user", "item"])["rating"].mean().reset_index()
    print(f'Shape of data after accounting for duplicate ratings:{df.shape}')

    user_features = df.groupby("user").agg(
        # new col name=('col_to_use', aggregation function)
        num_interactions=("item", "count"),
        num_unique_items=("item", "nunique"),

        mean_rating=("rating", "mean"),
        median_rating=("rating", "median"),
        std_rating=("rating", "std"),
        min_rating=("rating", "min"),
        max_rating=("rating", "max"),

        # fraction of ratings > 4
        high_rating_ratio=("rating", lambda x: (x >= 4).mean()),
        low_rating_ratio=("rating", lambda x: (x <= 2).mean()),

        # rating distribution
        rating_0_pct=("rating", lambda x: (x == 0).mean()),
        rating_1_pct=("rating", lambda x: (x == 1).mean()),
        rating_2_pct=("rating", lambda x: (x == 2).mean()),
        rating_3_pct=("rating", lambda x: (x == 3).mean()),
        rating_4_pct=("rating", lambda x: (x == 4).mean()),
        rating_5_pct=("rating", lambda x: (x == 5).mean()),

        user_rating_skew=("rating", lambda x: skew(x)),
        user_rating_kurt=("rating", lambda x: kurtosis(x)),
        rating_entropy=("rating", lambda x: entropy(x.value_counts(normalize=True)))
    ).reset_index()

    # fixes
    user_features["std_rating"] = user_features["std_rating"].fillna(0)
    user_features["user_rating_skew"] = user_features["user_rating_skew"].fillna(0)
    user_features["user_rating_kurt"] = user_features["user_rating_kurt"].fillna(0)

    user_features["interaction_density"] = (
        user_features["num_unique_items"] / 1000
    )

    return user_features

user_features_baseline = build_user_baseline(df)
user_features_baseline = user_features_baseline.drop(columns=['num_interactions'])
user_features_baseline = user_features_baseline.rename(columns={'num_unique_items': 'num_unique_items_rated'})
user_features_baseline = user_features_baseline.merge(
    labels,
    on="user",
    how="left"
)
user_features_baseline.head()


Shape of data before accounting for duplicate ratings: (177346, 3)
Shape of data after accounting for duplicate ratings:(177346, 3)


/var/folders/df/nqyr7tsx4xq5y1nsh200wt0w0000gn/T/ipykernel_40954/3860362287.py:35: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  user_rating_skew=("rating", lambda x: skew(x)),
/var/folders/df/nqyr7tsx4xq5y1nsh200wt0w0000gn/T/ipykernel_40954/3860362287.py:36: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  user_rating_kurt=("rating", lambda x: kurtosis(x)),


,user,num_unique_items_rated,mean_rating,median_rating,std_rating,min_rating,max_rating,high_rating_ratio,low_rating_ratio,rating_0_pct,rating_1_pct,rating_2_pct,rating_3_pct,rating_4_pct,rating_5_pct,user_rating_skew,user_rating_kurt,rating_entropy,interaction_density,label
0,100,88,3.147727,3.0,1.169928,0.0,5.0,0.431818,0.204545,0.045455,0.056818,0.102273,0.363636,0.363636,0.068182,-0.938904,0.737335,1.455461,0.088,0
1,101,231,3.406926,3.0,0.854016,0.0,5.0,0.432900,0.077922,0.008658,0.017316,0.051948,0.489177,0.346320,0.086580,-0.548093,1.864121,1.193838,0.231,0
2,102,98,3.673469,4.0,1.072385,1.0,5.0,0.683673,0.142857,0.000000,0.061224,0.081633,0.173469,0.489796,0.193878,-0.935300,0.403276,1.347083,0.098,0
3,103,103,3.000000,3.0,1.290994,0.0,5.0,0.417476,0.271845,0.058252,0.097087,0.116505,0.310680,0.349515,0.067961,-0.741755,-0.127405,1.555827,0.103,0
4,104,89,3.719101,4.0,0.988442,1.0,5.0,0.584270,0.056180,0.000000,0.044944,0.011236,0.359551,0.348315,0.235955,-0.622146,0.523690,1.265745,0.089,0


Now that we have our user behavioural features, we can start builing a baseline model.


In [29]:
# user_features.dtypes
# #  Separate Features and labels
y = user_features_baseline["label"]
X = user_features_baseline.drop(columns=["user", "label"])
X.isnull().sum()

# Since dataset is imbalanced we startify based on y
# to ensure both train and test set have the same anomaly ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
# Perform standardization of features
scaler = StandardScaler()

# fit -> learn parameters(mean, std) from data
# transform → apply transformation using those parameters
X_train_scaled = scaler.fit_transform(X_train)
# transform only no fitting occurs so that data leakage does not happen, 
# test data uses params of training data and tranforms
X_test_scaled = scaler.transform(X_test)



# Balanced weighting increases penalty for misclassifying anomalies.
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:,1]
print(y_prob[:5])

[0.08948964 0.94634002 0.93503102 0.2038238  0.03646232]


In [30]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual Normal", "Actual Anomaly"],
    columns=["Predicted Normal", "Predicted Anomaly"]
)
print("Confusion Matrix:")
print(cm_df)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_prob)
print("\nROC-AUC:", roc_auc)

# AUPRC (Average Precision Score)
auprc = average_precision_score(y_test, y_prob)
print("AUPRC:", auprc)

Confusion Matrix:
                Predicted Normal  Predicted Anomaly
Actual Normal                167                 33
Actual Anomaly                 8                 12

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.83      0.89       200
           1       0.27      0.60      0.37        20

    accuracy                           0.81       220
   macro avg       0.61      0.72      0.63       220
weighted avg       0.89      0.81      0.84       220


ROC-AUC: 0.8025
AUPRC: 0.5720286854485521


## Experimenting with baseline features using XGBoost

0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
dtype: float64

## Train XGBoost with Item and Interaction level features

In [ ]:
users = user_features_baseline["user"]
y = user_features_baseline["label"]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
roc_scores = []
pr_scores = []


for fold, (train_idx, val_idx) in enumerate(kf.split(users, y)):

    print(f"\nFold {fold+1}")

    train_users = users.iloc[train_idx]
    val_users = users.iloc[val_idx]

    # ---------- SPLIT RAW DATA ----------
    train_df = df[df["user"].isin(train_users)].copy()
    val_df   = df[df["user"].isin(val_users)].copy()

    # Compute item stats ONLY on train to prevent data leakage
    item_popularity = train_df.groupby("item").size()
    item_mean_rating = train_df.groupby("item")["rating"].mean()
    item_std_rating = train_df.groupby("item")["rating"].std()

    # apply stats to both train and val
    for d in [train_df, val_df]:
        d["item_popularity"] = d["item"].map(item_popularity)
        d["item_mean_rating"] = d["item"].map(item_mean_rating)
        d["item_std_rating"] = d["item"].map(item_std_rating)

    # handle missing values
    for d in [train_df, val_df]:
        d["item_popularity"] = d["item_popularity"].fillna(0)
        d["item_mean_rating"] = d["item_mean_rating"].fillna(d["rating"].mean())
        d["item_std_rating"] = d["item_std_rating"].fillna(1)
    
    # compute interaction level features
    for d in [train_df, val_df]:
        d["rating_deviation"] = d["rating"] - d["item_mean_rating"]
        d["normalized_deviation"] = d["rating_deviation"] / (d["item_std_rating"] + 1e-5)
        d["abs_deviation"] = d["rating_deviation"].abs()
    
    interaction_features_train = train_df.groupby("user").agg(
        mean_rating_deviation=("rating_deviation", "mean"),
        std_rating_deviation=("rating_deviation", "std"),
        mean_normalized_deviation=("normalized_deviation", "mean"),
        std_normalized_deviation=("normalized_deviation", "std"),
        mean_abs_deviation=("abs_deviation", "mean"),
        avg_item_popularity=("item_popularity", "mean"),
    ).reset_index()

    interaction_features_val = val_df.groupby("user").agg(
        mean_rating_deviation=("rating_deviation", "mean"),
        std_rating_deviation=("rating_deviation", "std"),
        mean_normalized_deviation=("normalized_deviation", "mean"),
        std_normalized_deviation=("normalized_deviation", "std"),
        mean_abs_deviation=("abs_deviation", "mean"),
        avg_item_popularity=("item_popularity", "mean"),
    ).reset_index()

    # merge with baseline features
    train_features = user_features_baseline[
        user_features_baseline["user"].isin(train_users)
    ].merge(interaction_features_train, on="user", how="left")

    val_features = user_features_baseline[
        user_features_baseline["user"].isin(val_users)
    ].merge(interaction_features_val, on="user", how="left")

    X_train = train_features.drop(columns=["user", "label"])
    y_train = train_features["label"]

    X_val = val_features.drop(columns=["user", "label"])
    y_val = val_features["label"]

    X_train = X_train.fillna(0)
    X_val = X_val.fillna(0)

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=10,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_val)[:, 1]

    roc = roc_auc_score(y_val, y_prob)
    pr = average_precision_score(y_val, y_prob)

    roc_scores.append(roc)
    pr_scores.append(pr)

    print(f"ROC-AUC: {roc:.4f}, PR-AUC: {pr:.4f}")


print("\nFinal Results:")
print("Mean ROC-AUC:", sum(roc_scores)/len(roc_scores))
print("Mean PR-AUC:", sum(pr_scores)/len(pr_scores))



Fold 1
ROC-AUC: 0.8417, PR-AUC: 0.6502

Fold 2
ROC-AUC: 0.8935, PR-AUC: 0.6313

Fold 3
ROC-AUC: 0.9140, PR-AUC: 0.7830

Fold 4
ROC-AUC: 0.8962, PR-AUC: 0.6865

Fold 5
ROC-AUC: 0.8718, PR-AUC: 0.6816

Final Results:
Mean ROC-AUC: 0.8834500000000001
Mean PR-AUC: 0.6865049074116021


## Training on entire dataset

In [ ]:
# Use full data
train_df = df.copy()
# compute item stats again
item_popularity = train_df.groupby("item").size()
item_mean = train_df.groupby("item")["rating"].mean()
item_std = train_df.groupby("item")["rating"].std()
# Build interaction features
train_df["item_popularity"] = train_df["item"].map(item_popularity)
train_df["item_mean"] = train_df["item"].map(item_mean)
train_df["item_std"] = train_df["item"].map(item_std)

train_df["rating_deviation"] = train_df["rating"] - train_df["item_mean"]
train_df["normalized_deviation"] = train_df["rating_deviation"] / (train_df["item_std"] + 1e-5)
train_df["abs_deviation"] = train_df["rating_deviation"].abs()

# Aggregate to user-level 
interaction_features = train_df.groupby("user").agg(
    mean_rating_deviation=("rating_deviation", "mean"),
    std_rating_deviation=("rating_deviation", "std"),
    mean_normalized_deviation=("normalized_deviation", "mean"),
    std_normalized_deviation=("normalized_deviation", "std"),
    mean_abs_deviation=("abs_deviation", "mean"),
    avg_item_popularity=("item_popularity", "mean"),
).reset_index()

# merge with user_level_features_baseline
full_features = user_features_baseline.merge(
    interaction_features,
    on="user",
    how="left"
).fillna(0)

X = full_features.drop(columns=["user", "label"])
y = full_features["label"]

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=10,
    random_state=42
)

model.fit(X, y)

#  Load test df
test_data = np.load('../input/first_batch.npz')
test_data = test_data['X']
test_df = pd.DataFrame(
    X_test,
    columns=["user","item","rating"]
)
test_df


[[2974    5    2]
 [2974   11    3]
 [2974   14    3]
 ...
 [3540  533    1]
 [3540  541    0]
 [3540  558    0]]
